In [ ]:
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.spatial import distance_matrix
from scipy.spatial import distance
from scipy.optimize import minimize
from scipy.optimize import basinhopping
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
import math
import shutil
import time
from numba import jit
import os
import statistics
from scipy.interpolate import interp1d
from scipy.interpolate import CubicSpline



In [ ]:
##############################################
# To Create your platelet, you can vary the 
# dimensions below. The rest of the data initialization
# section of the code does not need to be changed.
##############################################

flat_radius = 5.19  #flat plate radius

hex_radius = 5.19   #hexagon radius
# flat_radius = 6.5  #flat plate radius

# hex_radius = 6.5   #hexagon radius

theta = 61 *np.pi/180   #theta of double layer platelet system, in rad

n_rings = 3
z_dist = 5.44  #z distance between plateletes for 2 layer platelet system

x_dist = z_dist/np.tan(theta) #x distance shift of bottom layer based on theta
x_dist
##########

# percent = 0.775  #size percentage of the edge atom hexagon (<1 so the edge atoms have smaller radius)

flat_radius = 5.19  #flat plate radius

hex_radius = 5.19   #hexagon radius

theta = 61 *np.pi/180   #theta of double layer platelet system, in rad

n_rings = 3

x_dist = z_dist/np.tan(theta) #x distance shift of bottom layer based on theta
x_dist
y_dist=x_dist

x_dist

In [ ]:
def Plate(radius,w_max,h_max):
  w = radius
  h = np.sqrt(3)*radius/2

  coordinates = []
  for y in np.arange(0, h_max, h):
    f=0
    for x in np.arange(0, w_max, w):
      coordinates.append((x, y))
      f+=1
  coordinates = np.array(coordinates)
  
  i=0
  while i < (len(coordinates[:,0])+f*2):
    coordinates[i:i+f,0] += radius/2
    i += f*2
  return coordinates

###################


z_dist = 5.44

theta = 61 *np.pi/180   #theta of double layer platelet system, in rad

x_dist = z_dist/np.tan(theta) #x distance shift of bottom layer based on theta

flat_radius = 5.19


def hexa(n_rings,radius):
    #width and height of the hexagon
    w = radius*2/np.sqrt(3)
    h = radius
    N = int(n_rings/2)

    #generate empty arrays
    central_x = []
    central_y = []
    edge_x = []
    edge_y = []

    if (n_rings%2==1):
        print("n_rings must be even number")
    else:
        #top edge atom layer
        edge_x.append(0)
        edge_y.append(0)
        for i in range(N):
            edge_x.append(edge_x[-1]+1.5*w)
            edge_y.append(edge_y[-1])

        #first half of hexagon
        for i in range(N):
            central_x.append(edge_x[0]+0.75*w)
            central_y.append(edge_y[0]-0.5*h)
            for j in range(N -1 +i):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            central_x.append(edge_x[0])
            central_y.append(edge_y[0]-h)
            for j in range(N+i):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            edge_x.insert(0,edge_x[0]-0.75*w)
            edge_y.insert(0,edge_y[0]-1.5*h)
            central_x.append(edge_x[0]+1.5*w)
            central_y.append(edge_y[0])
            for j in range(N -1 +i):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            edge_x.append(central_x[-1]+1.5*w)
            edge_y.append(central_y[-1])

        #second half of hexagon
        for i in range(N-1):
            central_x.append(edge_x[0]+0.75*w)
            central_y.append(edge_y[0]-0.5*h)
            for j in range(N+(N-i)-1):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            central_x.append(edge_x[0]+1.5*w)
            central_y.append(edge_y[0]-h)
            for j in range(N+(N-i)-2):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            edge_x.insert(0,edge_x[0]+0.75*w)
            edge_y.insert(0,edge_y[0]-1.5*h)
            central_x.append(edge_x[0]+1.5*w)
            central_y.append(edge_y[0])
            for j in range(N +(N-i)-3):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            edge_x.append(central_x[-1]+1.5*w)
            edge_y.append(central_y[-1])
        central_x.append(edge_x[0]+0.75*w)
        central_y.append(edge_y[0]-0.5*h)
        for j in range(N):
            central_x.append(central_x[-1]+1.5*w)
            central_y.append(central_y[-1])
        central_x.append(edge_x[0]+1.5*w)
        central_y.append(edge_y[0]-h)
        for j in range(N-1):
            central_x.append(central_x[-1]+1.5*w)
            central_y.append(central_y[-1])
        edge_x.append(edge_x[0]+0.75*w)
        edge_y.append(edge_y[0]-1.5*h)
        for i in range(N):
            edge_x.append(edge_x[-1]+1.5*w)
            edge_y.append(edge_y[-1])

        #center of mass back to 0
        edge_x -= np.average(edge_x)
        edge_y -= np.average(edge_y)
        central_x -= np.average(central_x)
        central_y -= np.average(central_y)

        #NEW - MERGE EDGE AND CENTRAL
        central_x = np.concatenate((central_x,edge_x))
        central_y = np.concatenate((central_y,edge_y))

        #NEW - EDGE ATOMS
        edge_x = []
        edge_y = []

        edge_x.append(n_rings*0.75*w+0.5*w)
        edge_y.append(0)

        edge_x.append(edge_x[-1]-0.5*w)
        edge_y.append(edge_y[-1]-h)
        for i in range(N-1):
            edge_x.append(edge_x[-1]-0.75*w)
            edge_y.append(edge_y[-1]-1.5*h)
        edge_x.append(edge_x[-1]-0.5*w)
        edge_y.append(edge_y[-1]-h)

        edge_x.append(edge_x[-1]-w)
        edge_y.append(edge_y[-1])
        for i in range(N-1):
            edge_x.append(edge_x[-1]-1.5*w)
            edge_y.append(edge_y[-1])
        edge_x.append(edge_x[-1]-w)
        edge_y.append(edge_y[-1])

        edge_x.append(edge_x[-1]-0.5*w)
        edge_y.append(edge_y[-1]+h)
        for i in range(N-1):
            edge_x.append(edge_x[-1]-0.75*w)
            edge_y.append(edge_y[-1]+1.5*h)
        edge_x.append(edge_x[-1]-0.5*w)
        edge_y.append(edge_y[-1]+h)

        edge_x.append(edge_x[-1]+0.5*w)
        edge_y.append(edge_y[-1]+h)
        for i in range(N-1):
            edge_x.append(edge_x[-1]+0.75*w)
            edge_y.append(edge_y[-1]+1.5*h)
        edge_x.append(edge_x[-1]+0.5*w)
        edge_y.append(edge_y[-1]+h)

        edge_x.append(edge_x[-1]+w)
        edge_y.append(edge_y[-1])
        for i in range(N-1):
            edge_x.append(edge_x[-1]+1.5*w)
            edge_y.append(edge_y[-1])
        edge_x.append(edge_x[-1]+w)
        edge_y.append(edge_y[-1])

        edge_x.append(edge_x[-1]+0.5*w)
        edge_y.append(edge_y[-1]-h)
        for i in range(N-1):
            edge_x.append(edge_x[-1]+0.75*w)
            edge_y.append(edge_y[-1]-1.5*h)

    return central_x,central_y,edge_x,edge_y

In [ ]:

y,x ,yy, xx = hexa(2,hex_radius) 
#plot hexagonal flat plate
plt.figure(figsize=(7,7))
plt.plot(x,y,"o")
plt.plot(xx,yy,"o")
# plt.xlim([-25,25])
# plt.ylim([-25,25])
# plt.ylim([10,20])
# plt.xlim([10,20])

plt.grid()
plt.show()

#print total number of atoms
print("Total number of atoms: ",(len(x)+len(xx)))
print("Long diameter: %.3f A: " %(max(yy)-min(yy)))

z = np.zeros(len(x))
zz = np.zeros(len(xx))

#put x,y,z data into one array
CentralPlatelet = np.stack((x,y,z),axis=-1)
EdgePlatelet = np.stack((xx,yy,zz),axis=-1)

#make two platelets
CentralPlatelet2=CentralPlatelet.copy()
CentralPlatelet2[:,2]=CentralPlatelet[:,2]+z_dist
CentralPlatelet2[:,0]=CentralPlatelet[:,0]+x_dist

EdgePlatelet2=EdgePlatelet.copy()
EdgePlatelet2[:,2]=EdgePlatelet[:,2]+z_dist
EdgePlatelet2[:,0]=EdgePlatelet[:,0]+x_dist





#save into one array
CentralPlate = np.vstack((CentralPlatelet,CentralPlatelet2))
EdgePlate = np.vstack((EdgePlatelet,EdgePlatelet2))

CentralPlate[:,0] = CentralPlate[:,0] + 6.71 + abs(min(xx))
EdgePlate[:,0] = EdgePlate[:,0] + 6.71 + abs(min(xx))
CentralPlate[:,1] = CentralPlate[:,1] + 8.6 + abs(min(yy))
EdgePlate[:,1] = EdgePlate[:,1] + 8.6 + abs(min(yy))
print(min(EdgePlate[:,0]))
print(min(EdgePlate[:,1]))

plt.figure(figsize=(7,7))
plt.plot(CentralPlate[:,0],CentralPlate[:,1],"o")
plt.plot(EdgePlate[:,0],EdgePlate[:,1],"o")


plt.grid()
plt.show()




#plot together
fig = plt.figure(figsize=(10,6))
ax = fig.add_subplot(projection='3d')
ax.scatter(CentralPlate[:,0],CentralPlate[:,1],CentralPlate[:,2],label="Central Atoms")
ax.scatter(EdgePlate[:,0],EdgePlate[:,1],EdgePlate[:,2],label="Edge Atoms")
ax.set_xlabel('X [Å]',labelpad=12,fontsize=18)
ax.set_ylabel('Y [Å]',labelpad=10,fontsize=18)
ax.set_zlabel('Z [Å]',labelpad=7,fontsize=18)
ax.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9),fontsize=18)
plt.show()
print("Total number of edge atoms, double platelet: ",len(xx)*2)
print("Total number of central atoms, double platelet: ",len(x)*2)

print("Long diameter, large platelet: %.3f A: " %(max(xx)-min(xx)))


In [ ]:





coord = Plate(flat_radius,41.53,53) #  Original Platelet
print(max(coord[:,0]))



x_central = coord[:, 0]
y_central = coord[:, 1]
z_central = np.zeros(len(x_central))


Flat_central = np.stack((x_central, y_central, z_central), axis=-1)
Flat2_central = Flat_central.copy()
Flat2_central[:, 2] += z_dist
Flat2_central[:, 0] += x_dist




fig,ax = plt.subplots(figsize=(10,8))
ax.xaxis.set_major_locator(MultipleLocator(2.595))
ax.yaxis.set_major_locator(MultipleLocator(4.4946))
ax.grid(which='major', color='#CCCCCC', linestyle='--')
plt.plot(Flat_central[:,0],Flat_central[:,1],"o")
plt.show()
print("Number of atoms:",len(coord[:,0]))
print("Area:",(max(coord[:,0])*max(coord[:,1])))

fig,ax = plt.subplots(figsize=(10,8))
ax.xaxis.set_major_locator(MultipleLocator(2.595))
ax.yaxis.set_major_locator(MultipleLocator(4.4946))
ax.grid(which='major', color='#CCCCCC', linestyle='--')
plt.plot(Flat2_central[:,0],Flat2_central[:,1],"o")
plt.show()


FlatPlate = np.vstack((Flat_central, Flat2_central))

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(FlatPlate[:,0],FlatPlate[:,1],FlatPlate[:,2])


ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
plt.show()

plt.plot(FlatPlate[:,0],FlatPlate[:,2],"o")

plt.grid()
plt.show()


FlatPlate = np.vstack((Flat_central, Flat2_central))


fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(FlatPlate[:, 0], FlatPlate[:, 1], FlatPlate[:, 2] , s=10, alpha=0.5)
ax.scatter(CentralPlate[:,0],CentralPlate[:,1],CentralPlate[:,2]+10,label="Central Atoms")
ax.scatter(EdgePlate[:,0],EdgePlate[:,1],EdgePlate[:,2]+10,label="Edge Atoms")

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Mirrored Platelets in X and Y Directions (3D)')
ax.view_init(elev=90, azim=270)

plt.show()


In [ ]:
# Creating periodic sheet
x_offsets = [-2 * 46.72, -1 * 46.72, 0, 1 * 46.72, 2 * 46.72]  
y_offsets = [-2 * 53.94, -1 * 53.94, 0, 1 * 53.94, 2 * 53.94]  



FlatPlate_1 = []
FlatPlate_2 = []

for x in x_offsets:
    for y in y_offsets:
        displacement = np.array([x, y, 0])  # Displacement in x and y directions
        FlatPlate_1.append(Flat_central + displacement)
        FlatPlate_2.append(Flat2_central + displacement)

# Convert lists to numpy arrays
FlatPlate_1 = np.vstack(FlatPlate_1)
FlatPlate_2 = np.vstack(FlatPlate_2)


FlatPlate_P = np.vstack((FlatPlate_1, FlatPlate_2))

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(FlatPlate_P[:, 0], FlatPlate_P[:, 1], FlatPlate_P[:, 2] , s=10, alpha=0.5)
ax.scatter(CentralPlate[:,0],CentralPlate[:,1],CentralPlate[:,2]+10,label="Central Atoms")
ax.scatter(EdgePlate[:,0],EdgePlate[:,1],EdgePlate[:,2]+10,label="Edge Atoms")

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Mirrored Platelets in X and Y Directions (3D)')
ax.view_init(elev=0, azim=0)

plt.show()


In [ ]:
#Crearting periodic hexagons and plotting togheder in a way that match with my atomistic simulations

x_H = [-2 * 41.52, -1 * 41.52, 0, 1 * 41.52, 2 * 41.52]  
y_H = [-2 * 44.95, -1 * 44.95, 0, 1 * 44.95, 2 * 44.95]  



Edge_1 = []
Center_1 = []

for x in x_offsets:
    for y in y_offsets:
        displacement = np.array([x, y, 0])  # Displacement in x and y directions
        Edge_1.append(EdgePlate + displacement)
        Center_1.append(CentralPlate + displacement)

# Convert lists to numpy arrays
Edge_1_whole = np.vstack(Edge_1)
Center_1_whole = np.vstack(Center_1)

Edge_2_whole = np.vstack(Edge_1)
Center_2_whole = np.vstack(Center_1)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(FlatPlate_P[:, 0], FlatPlate_P[:, 1] , s=10, alpha=0.5)
ax.scatter(Center_1_whole[:,0],Center_1_whole[:,1] , s=10 ,label="Moved")
ax.scatter(Edge_1_whole[:,0],Edge_1_whole[:,1], s=10 , label="Moved")
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Mirrored Platelets in X and Y Directions (2D)')
ax.legend()
plt.grid()
plt.show()




In [ ]:
# separating the main hexagon and images to prepare them for the distance calculations 
# (I could have exclude them in the previous cell)

############## This is for pairwise distances between original hexagon and periodic sheet



# Boundary definitions
x_min_1 = 0
x_max_1 = 41.52  
y_min_1 = 0
y_max_1 = 44.95

# Filter for MAIN REGION (points INSIDE boundaries)
mask_1_ed = (
    (Edge_1_whole[:, 0] >= x_min_1) & (Edge_1_whole[:, 0] <= x_max_1) & 
    (Edge_1_whole[:, 1] >= y_min_1) & (Edge_1_whole[:, 1] <= y_max_1)
)

Edge_filtered_main = Edge_1_whole[mask_1_ed]

mask_1_cen = (
    (Center_1_whole[:, 0] >= x_min_1) & (Center_1_whole[:, 0] <= x_max_1) & 
    (Center_1_whole[:, 1] >= y_min_1) & (Center_1_whole[:, 1] <= y_max_1)
)

Center_filtered_main = Center_1_whole[mask_1_cen]

# Filter for EXTERNAL REGION (points OUTSIDE boundaries)
mask_1_ed_ex = (
    (Edge_2_whole[:, 0] < x_min_1) | (Edge_2_whole[:, 0] > x_max_1) | 
    (Edge_2_whole[:, 1] < y_min_1) | (Edge_2_whole[:, 1] > y_max_1))


mask_1_cen_ex = (
    (Center_2_whole[:, 0] < x_min_1) | (Center_2_whole[:, 0] > x_max_1) |
    (Center_2_whole[:, 1] < y_min_1) | (Center_2_whole[:, 1] > y_max_1))


Edge_filtered_ex = Edge_2_whole[mask_1_ed_ex]
Center_filtered_ex = Center_2_whole[mask_1_cen_ex]


plt.figure(figsize=(8,6))
plt.scatter(Edge_filtered_main[:,0], Edge_filtered_main[:,1], label='Edge Particles')
plt.scatter(Center_filtered_main[:,0], Center_filtered_main[:,1], label='Center Particles')
plt.axvline(x_min_1, color='r', linestyle='--')
plt.axvline(x_max_1, color='r', linestyle='--')
plt.axhline(y_min_1, color='b', linestyle='--')
plt.axhline(y_max_1, color='b', linestyle='--')
plt.scatter(Edge_filtered_ex[:,0], Edge_filtered_ex[:,1], label='Edge Particles')
plt.scatter(Center_filtered_ex[:,0], Center_filtered_ex[:,1], label='Center Particles')

plt.legend()
plt.title('Particles Outside Boundary Box')
plt.show()




In [ ]:
# Creating a sheet with the same size as atomistic simulation box (This is used for the distance calculations)



coord_org = Plate(flat_radius,42,53) #  Original Platelet
print(max(coord_org[:,0]))
x_central_org = coord_org[:, 0]
y_central_org = coord_org[:, 1]
z_central_org = np.zeros(len(x_central_org))


Flat_central_org = np.stack((x_central_org, y_central_org, z_central_org), axis=-1)
Flat2_central_org = Flat_central_org.copy()
Flat2_central_org[:, 2] += z_dist
Flat2_central_org[:, 0] += x_dist


# fig,ax = plt.subplots(figsize=(10,8))
# ax.xaxis.set_major_locator(MultipleLocator(2.595))
# ax.yaxis.set_major_locator(MultipleLocator(4.4946))
# ax.grid(which='major', color='#CCCCCC', linestyle='--')
# plt.plot(Flat_central_org[:,0],Flat_central_org[:,1],"o")
# plt.show()
# print("Number of atoms:",len(coord_org[:,0]))
# print("Area:",(max(coord_org[:,0])*max(coord_org[:,1])))

# fig,ax = plt.subplots(figsize=(10,8))
# ax.xaxis.set_major_locator(MultipleLocator(2.595))
# ax.yaxis.set_major_locator(MultipleLocator(4.4946))
# ax.grid(which='major', color='#CCCCCC', linestyle='--')
# plt.plot(Flat2_central_org[:,0],Flat2_central_org[:,1],"o")
# plt.show()

FlatPlate_main = np.vstack((Flat_central_org, Flat2_central_org))
#############

x_min_1 = 0
x_max_1 = 41.52  
y_min_1 = 0
y_max_1 = 47

# Filter for Layer1 (points INSIDE boundaries)
mask_1_F1 = (
    (Flat_central_org[:, 0] >= x_min_1) & (Flat_central_org[:, 0] <= x_max_1) & 
    (Flat_central_org[:, 1] >= y_min_1) & (Flat_central_org[:, 1] <= y_max_1)
)

F1_filtered_main = Flat_central_org[mask_1_F1]


# Filter for Layer2 (points INSIDE boundaries)

x_min_1 = 2.6
x_max_1 = 46.71  
y_min_1 = 0
y_max_1 = 47

# Filter for Layer1 (points INSIDE boundaries)
mask_2_F2 = (
    (Flat2_central_org[:, 0] >= x_min_1) & (Flat2_central_org[:, 0] <= x_max_1) & 
    (Flat2_central_org[:, 1] >= y_min_1) & (Flat2_central_org[:, 1] <= y_max_1)
)

F2_filtered_main = Flat2_central_org[mask_2_F2]


###########

FlatPlate_main_Filtered = np.vstack((F1_filtered_main, F2_filtered_main))



ig,ax = plt.subplots(figsize=(10,8))
ax.xaxis.set_major_locator(MultipleLocator(2.595))
ax.yaxis.set_major_locator(MultipleLocator(4.4946))
ax.grid(which='major', color='#CCCCCC', linestyle='--')
plt.plot(F1_filtered_main[:,0],F1_filtered_main[:,1],"o")
plt.show()

fig,ax = plt.subplots(figsize=(10,8))
ax.xaxis.set_major_locator(MultipleLocator(2.595))
ax.yaxis.set_major_locator(MultipleLocator(4.4946))
ax.grid(which='major', color='#CCCCCC', linestyle='--')
plt.plot(F2_filtered_main[:,0],F2_filtered_main[:,1],"o")
plt.show()




In [ ]:
from pathlib import Path

DATA_DIR = Path("../../data/training/fully_periodic_sheet_small_hex")

f_Stack = np.genfromtxt(
    DATA_DIR / "dspacing_PMF.out",
    dtype=float,
    skip_header=0,
    skip_footer=0
)

sorted_indices = np.argsort(f_Stack[:, 0])
f_Stack = f_Stack[sorted_indices]
d_spacing = f_Stack[:, 0]
z_S = f_Stack[:, 0]


# 1. Distances between the original hexagon and periodic sheet within 25 Å
# 2. Distances between original sheet (not periodic one) and imaged hexagons

cen_dis = {}

for i, spacing in enumerate(z_S):

    # 1. Distances between original hexagon centers and periodic sheet
    Hex_cen = Center_filtered_main.copy()
    Hex_cen[:, 2] += spacing

    dist_matrix = distance.cdist(
        Hex_cen,
        FlatPlate_P
    )
    final_distances = dist_matrix[
        dist_matrix <= 25
    ]

    # 2. Distances between original sheet and periodic hexagon centers
    Hex_cen_ex = Center_filtered_ex.copy()
    Hex_cen_ex[:, 2] += spacing

    dist_matrix_2 = distance.cdist(
        FlatPlate_main_Filtered,
        Hex_cen_ex
    )
    final_distances_2 = dist_matrix_2[
        dist_matrix_2 <= 25
    ]

    # Combine distances
    combined_distances = np.concatenate(
        (final_distances, final_distances_2)
    )

    cen_dis[round(spacing, 5)] = combined_distances

    print(
        f"Iteration {i}: d-spacing = {spacing}, "
        f"Kept {len(combined_distances)} distances"
    )


# ------------------------------------------------------------------
# Save Center-Center pairwise distances
# ------------------------------------------------------------------

output_file_npy = DATA_DIR / "PairDis_Infhex_1_CC.npy"
np.save(output_file_npy, cen_dis, allow_pickle=True)

# Load and check stored distances
loaded_data = np.load(
    output_file_npy,
    allow_pickle=True
).item()

# Compare d-spacing values between .out and .npy
d_spacing_out = {
    round(x, 5) for x in f_Stack[:, 0]
}
d_spacing_npy = {
    round(x, 5) for x in loaded_data.keys()
}

missing_in_out = d_spacing_npy - d_spacing_out
missing_in_npy = d_spacing_out - d_spacing_npy

print(f"Missing in .out file: {missing_in_out}")
print(f"Missing in .npy file: {missing_in_npy}")

# Check stored distances for first five d-spacing values
for key in list(loaded_data.keys())[:5]:
    print(
        f"d-spacing: {key}, "
        f"Number of distances: {len(loaded_data[key])}"
    )


# ------------------------------------------------------------------
# Edge-Center pairwise distances
# ------------------------------------------------------------------

ed_cen_dis = {}

for i, spacing in enumerate(z_S):

    Hex_ed = Edge_filtered_main.copy()
    Hex_ed[:, 2] += spacing

    dist_matrix = distance.cdist(
        Hex_ed,
        FlatPlate_P
    )
    final_distances = dist_matrix[
        dist_matrix <= 25
    ]

    Hex_ed_ex = Edge_filtered_ex.copy()
    Hex_ed_ex[:, 2] += spacing

    dist_matrix_2 = distance.cdist(
        FlatPlate_main_Filtered,
        Hex_ed_ex
    )
    final_distances_2 = dist_matrix_2[
        dist_matrix_2 <= 25
    ]

    # Combine distances
    combined_distances = np.concatenate(
        (final_distances, final_distances_2)
    )

    ed_cen_dis[round(spacing, 5)] = combined_distances

    print(
        f"Iteration {i}: d-spacing = {spacing}, "
        f"Kept {len(combined_distances)} distances"
    )


# ------------------------------------------------------------------
# Save Edge-Center pairwise distances
# ------------------------------------------------------------------

output_file_npy = DATA_DIR / "PairDis_Infhex_1_EC.npy"
np.save(output_file_npy, ed_cen_dis, allow_pickle=True)

# Load and check stored distances
loaded_data = np.load(
    output_file_npy,
    allow_pickle=True
).item()

# Compare d-spacing values between .out and .npy
d_spacing_out = {
    round(x, 5) for x in f_Stack[:, 0]
}
d_spacing_npy = {
    round(x, 5) for x in loaded_data.keys()
}

missing_in_out = d_spacing_npy - d_spacing_out
missing_in_npy = d_spacing_out - d_spacing_npy

print(f"Missing in .out file: {missing_in_out}")
print(f"Missing in .npy file: {missing_in_npy}")

# Check stored distances for first five d-spacing values
for key in list(loaded_data.keys())[:5]:
    print(
        f"d-spacing: {key}, "
        f"Number of distances: {len(loaded_data[key])}"
    )